# Reproducing the pruning-based bias experiment

This notebook walks through the pipeline from the paper "Breaking Down Bias: On The Limits of Generalizable Pruning Strategies". It mirrors the local CLI script in [reproduce_experiment.py](reproduce_experiment.py) and focuses on the purchase benchmark plus cross-context transfer.

The workflow is organized into three phases:

- component selection from named prompt families
- pruning with neuron hooks and attention-head masks
- evaluation, summary tables, and plots

In [ ]:
from pathlib import Path
import json

import numpy as np
import torch

import reproduce_experiment as rep

# Notebook-level run options
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
SMOKE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
USE_SMOKE_TEST = True
OUTPUT_DIR = Path("notebook_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Environment and Dependency Setup

Install or import `torch`, `transformers`, `numpy`, `scipy`, and `matplotlib`. The notebook uses the same deterministic settings and device selection logic as the script, and it can switch between the smaller smoke-test model and the full `Meta-Llama-3-8B-Instruct` checkpoint.

In [ ]:
import random

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)

MODEL_TO_USE = SMOKE_MODEL_NAME if USE_SMOKE_TEST else MODEL_NAME
print(f"Using model: {MODEL_TO_USE}")

## 2. Define Prompt Specifications and Group Name Lists

Load the paper's prompt families and the minority/majority name lists used for group-wise evaluation. The local script keeps the final prompt sets directly in code so the notebook can reuse them without duplication.

In [ ]:
PromptSpec = rep.PromptSpec
PURCHASE_PROMPTS = rep.PURCHASE_PROMPTS
ACTIVITY_PROMPTS = rep.ACTIVITY_PROMPTS
SERVICE_PROMPTS = rep.SERVICE_PROMPTS
FINANCE_PROMPTS = rep.FINANCE_PROMPTS
BLACK_NAMES = rep.BLACK_NAMES
WHITE_NAMES = rep.WHITE_NAMES

print(f"Purchase prompts: {len(PURCHASE_PROMPTS)}")
print(f"Activity prompts: {len(ACTIVITY_PROMPTS)}")
print(f"Service prompts: {len(SERVICE_PROMPTS)}")
print(f"Finance prompts: {len(FINANCE_PROMPTS)}")

## 3. Implement Numeric Parsing and Bias Metrics

Reuse the paper-aligned helpers for numeric extraction, winsorization, pooled standard deviation, Standardized Mean Difference, Earth Mover's Distance, utility bounds, and inlier ratio.

In [ ]:
parse_numeric = rep.parse_numeric
winsorize = rep.winsorize
pooled_standard_deviation = rep.pooled_standard_deviation
standardized_mean_difference = rep.standardized_mean_difference
earth_movers_distance = rep.earth_movers_distance
compute_utility_bounds = rep.compute_utility_bounds
inlier_ratio = rep.inlier_ratio

example_outputs = ["$125.00", "about 180 dollars", "72%"]
parsed = [parse_numeric(text) for text in example_outputs]
parsed

## 4. Implement Token Span Detection and Attention/Neuron Scoring

Use the local reproduction helpers to find the name span in tokenized prompts, capture MLP activations, and score attention heads by their influence on the group-name token region.

In [ ]:
prompt_name_span = rep.prompt_name_span
score_prompt_components = rep.score_prompt_components
aggregate_group_scores = rep.aggregate_group_scores

# The helpers below depend on a loaded model/tokenizer pair.
# They are called later in the notebook after the model is initialized.

## 5. Aggregate Group Component Scores per Prompt Variation

This section computes per-name component scores and averages them across Black-associated and white-associated name groups for each prompt variation.

In [ ]:
def load_model_pair(model_name: str):
    return rep.load_model(model_name)

# Example wiring for a future scoring pass:
# model, tokenizer = load_model_pair(MODEL_TO_USE)
# prompt_scores = aggregate_group_scores(model, tokenizer, PURCHASE_PROMPTS[0], BLACK_NAMES)
# prompt_scores

## 6. Select Biased Components (Neurons and Heads)

Rank the minority and majority scores separately, then apply the paper's thresholded set-difference rule to keep the minority-specific components that are not in the top majority list.

In [ ]:
select_biased_components = rep.select_biased_components
run_selection_pass = rep.run_selection_pass
intersection_components = rep.intersection_components

# Thresholds used by the paper
TAU_NEURON_MIN = 0.40
TAU_NEURON_MAJ = 0.35
TAU_HEAD_MIN = 40
TAU_HEAD_MAJ = 5

## 7. Apply Pruning via Neuron Hooks and Head Masks

Build the reusable pruning utilities used by the notebook and the script: indexed component maps, neuron-masking hooks, and attention-head mask tensors.

In [ ]:
component_index_sets = rep.component_index_sets
NeuronMaskContext = rep.NeuronMaskContext
build_head_mask = rep.build_head_mask

# These helpers are applied later when evaluating pruned generations.

## 8. Generate Model Outputs and Evaluate Fairness/Utility Metrics

Generate deterministic outputs for both groups, compute baseline utility bounds from the unpruned model, and evaluate SMD, EMD, inlier ratio, and group means after pruning.

In [ ]:
evaluate_prompt_family = rep.evaluate_prompt_family

# Example:
# model, tokenizer = load_model_pair(MODEL_TO_USE)
# metrics = evaluate_prompt_family(model, tokenizer, PURCHASE_PROMPTS[0], [], max_new_tokens=32)
# metrics

## 9. Run Prompt-Specific and Leave-One-Out Purchase Experiments

Execute selection on the purchase prompt family, evaluate each prompt variation with prompt-specific pruning, and then run the within-context leave-one-out experiment using intersections of the remaining variations.

In [ ]:
# Suggested execution pattern:
# model, tokenizer = load_model_pair(MODEL_TO_USE)
# purchase_selection = run_selection_pass(
#     model,
#     tokenizer,
#     PURCHASE_PROMPTS,
#     tau_neuron_min=TAU_NEURON_MIN,
#     tau_neuron_maj=TAU_NEURON_MAJ,
#     tau_head_min=TAU_HEAD_MIN,
#     tau_head_maj=TAU_HEAD_MAJ,
# )
# purchase_selection

## 10. Run Cross-Context Selection and Purchase Transfer Evaluation

Select components from the Activity, Service, and Finance prompt families, intersect the selected sets, and then transfer the pruned components back onto the purchase variations to compare neuron and head pruning behavior.

In [ ]:
# cross_context_prompts = ACTIVITY_PROMPTS + SERVICE_PROMPTS + FINANCE_PROMPTS
# cross_selection = run_selection_pass(
#     model,
#     tokenizer,
#     cross_context_prompts,
#     tau_neuron_min=TAU_NEURON_MIN,
#     tau_neuron_maj=TAU_NEURON_MAJ,
#     tau_head_min=TAU_HEAD_MIN,
#     tau_head_maj=TAU_HEAD_MAJ,
# )
# cross_neuron = intersection_components([cross_selection[p.variation]["neuron"] for p in cross_context_prompts])
# cross_head = intersection_components([cross_selection[p.variation]["head"] for p in cross_context_prompts])
# cross_neuron, cross_head

## 11. Build Summary Tables and Save JSON Artifacts

Aggregate mean metrics across prompt-specific, leave-one-out, and cross-context blocks, then persist the complete results dictionary and the compact summary to JSON files.

In [ ]:
build_summary = rep.build_summary
save_json = rep.save_json

# Example:
# results = rep.run_full_reproduction(
#     argparse.Namespace(
#         model_name=MODEL_TO_USE,
#         device_map="auto",
#         torch_dtype="auto",
#         max_new_tokens=32,
#         tau_neuron_min=TAU_NEURON_MIN,
#         tau_neuron_maj=TAU_NEURON_MAJ,
#         tau_head_min=TAU_HEAD_MIN,
#         tau_head_maj=TAU_HEAD_MAJ,
#         output_json=str(OUTPUT_DIR / "results.json"),
#         plot_dir=str(OUTPUT_DIR / "plots"),
#     )
# )
# results["summary"]

## 12. Plot SMD and Inlier-Ratio Results

Create the three main figure families from the paper-style pipeline: prompt-specific SMD/inlier plots, leave-one-out SMD plots, and cross-context purchase SMD plots. Figures are written to the configurable output directory.

In [ ]:
plot_results = rep.plot_results

# If you already have a results dictionary, call:
# plot_results(results, OUTPUT_DIR / "plots")